In [1]:
import os 
from pathlib import Path 
import polars as pl
import pandas as pd

DATA_PATH = Path('/data/gusev/PROFILE/CLINICAL/OncDRS/ALL_2025_03/')

In [2]:
from pathlib import Path
from time import perf_counter

import polars as pl


# ============================================================
# Column definitions
# ============================================================

ehr_cols = ["DFCI_MRN", "START_DT", "DIAGNOSIS_ICD10_CD", "DIAGNOSIS_ICD10_CD2", "DIAGNOSIS_ICD10_CD3"]
meds_cols = ["DFCI_MRN", "NCI_PREFERRED_MED_NM", "MED_START_DT"]
status_cols = ["DFCI_MRN", "BIRTH_DT", "GENDER_NM", "HYBRID_DEATH_DT", "DERIVED_LAST_ALIVE_DATE"]
labs_cols = ["DFCI_MRN", "SPECIMEN_COLLECT_DT", "TEST_TYPE_CD", "TEST_TYPE_DESCR", 
             "RESULT_TYPE_DESCR", "NUMERIC_RESULT", "TEXT_RESULT", "RESULT_UOM_NM"]
icd_cols = ["DIAGNOSIS_ICD10_CD", "DIAGNOSIS_ICD10_CD2", "DIAGNOSIS_ICD10_CD3"]

# ============================================================
# Polars workflow
# ============================================================

polars_start = perf_counter()


# ------------------------------------------------------------
# 1. Lazily read the diagnosis file
# ------------------------------------------------------------

ehr_lf = (
    pl.scan_csv(
        DATA_PATH / "EHR_DIAGNOSIS.csv",
        schema_overrides={
            "DFCI_MRN": pl.String,
            "DIAGNOSIS_ICD10_CD": pl.String,
            "DIAGNOSIS_ICD10_CD2": pl.String,
            "DIAGNOSIS_ICD10_CD3": pl.String,
        },
    )
    .select(ehr_cols)
)


# ------------------------------------------------------------
# 2. Identify unique prostate-cancer MRNs
# ------------------------------------------------------------

prostate_mrns_pl = (
    ehr_lf
    .filter(
        pl.any_horizontal(
            [
                pl.col("DIAGNOSIS_ICD10_CD") == "C61",
                pl.col("DIAGNOSIS_ICD10_CD2") == "C61",
                pl.col("DIAGNOSIS_ICD10_CD3") == "C61",
            ]
        )
    )
    .select("DFCI_MRN")
    .drop_nulls()
    .unique()
    .collect(streaming=True)
)

print(f"Unique prostate MRNs: {prostate_mrns_pl.height:,}")


# Convert back to a LazyFrame for semi-joins
prostate_mrns_lf = prostate_mrns_pl.lazy()


# ------------------------------------------------------------
# 3. Lazily read and filter medications
# ------------------------------------------------------------

meds_lf = (
    pl.scan_csv(
        DATA_PATH / "MEDICATIONS.csv",
        schema_overrides={
            "DFCI_MRN": pl.String,
        },
    )
    .select(meds_cols)
    .join(
        prostate_mrns_lf,
        on="DFCI_MRN",
        how="semi",
    )
)


# ------------------------------------------------------------
# 4. Lazily read and filter patient status
# ------------------------------------------------------------

status_lf = (
    pl.scan_csv(
        DATA_PATH / "PT_INFO_STATUS_REGISTRATION.csv",
        schema_overrides={
            "DFCI_MRN": pl.String,
        },
    )
    .select(status_cols)
    .join(
        prostate_mrns_lf,
        on="DFCI_MRN",
        how="semi",
    )
)


# ------------------------------------------------------------
# 5. Lazily read and filter labs
# ------------------------------------------------------------

labs_lf = (
    pl.scan_csv(
        DATA_PATH / "OUTPT_LAB_RESULTS_LABS.csv",
        schema_overrides={
            "DFCI_MRN": pl.String,
        },
    )
    .select(labs_cols)
    .join(
        prostate_mrns_lf,
        on="DFCI_MRN",
        how="semi",
    )
)


# ------------------------------------------------------------
# 6. Execute each query using the older streaming syntax
# ------------------------------------------------------------

meds_df_pl = meds_lf.collect(streaming=True)

status_df_pl = status_lf.collect(streaming=True)

labs_df_pl = labs_lf.collect(streaming=True)


# ------------------------------------------------------------
# 7. Timing and result summary
# ------------------------------------------------------------

polars_seconds = perf_counter() - polars_start

print()
print(f"Polars total time: {polars_seconds:,.2f} seconds")
print(f"Polars total time: {polars_seconds / 60:,.2f} minutes")
print()
print(f"Unique prostate MRNs: {prostate_mrns_pl.height:,}")
print(f"Medication rows:       {meds_df_pl.height:,}")
print(f"Status rows:           {status_df_pl.height:,}")
print(f"Lab rows:              {labs_df_pl.height:,}")

Unique prostate MRNs: 11,169

Polars total time: 28.69 seconds
Polars total time: 0.48 minutes

Unique prostate MRNs: 11,169
Medication rows:       2,601,597
Status rows:           11,169
Lab rows:              7,312,260


In [ ]:
from time import perf_counter

import pandas as pd


# ============================================================
# pandas/PyArrow benchmark
# ============================================================

pandas_start = perf_counter()


ehr_df_pd = pd.read_csv(
    DATA_PATH / "EHR_DIAGNOSIS.csv",
    usecols=ehr_cols,
    engine="pyarrow",
    dtype_backend="pyarrow",
    dtype={
        "DFCI_MRN": "string[pyarrow]",
        "DIAGNOSIS_ICD10_CD": "string[pyarrow]",
        "DIAGNOSIS_ICD10_CD2": "string[pyarrow]",
        "DIAGNOSIS_ICD10_CD3": "string[pyarrow]",
    },
)


# Identify rows where any diagnosis column is C61
prostate_mask_pd = (
    ehr_df_pd[icd_cols]
    .eq("C61")
    .any(axis=1)
)


# Create a set because pandas membership testing is efficient against it
prostate_mrns_pd = set(
    ehr_df_pd.loc[
        prostate_mask_pd,
        "DFCI_MRN",
    ]
    .dropna()
    .unique()
    .tolist()
)


# Diagnosis data is no longer required
del ehr_df_pd
del prostate_mask_pd


meds_df_pd = pd.read_csv(
    DATA_PATH / "MEDICATIONS.csv",
    usecols=meds_cols,
    engine="pyarrow",
    dtype_backend="pyarrow",
    dtype={
        "DFCI_MRN": "string[pyarrow]",
    },
)

meds_df_pd = meds_df_pd.loc[
    meds_df_pd["DFCI_MRN"].isin(prostate_mrns_pd)
].reset_index(drop=True)


status_df_pd = pd.read_csv(
    DATA_PATH / "PT_INFO_STATUS_REGISTRATION.csv",
    usecols=status_cols,
    engine="pyarrow",
    dtype_backend="pyarrow",
    dtype={
        "DFCI_MRN": "string[pyarrow]",
    },
)

status_df_pd = status_df_pd.loc[
    status_df_pd["DFCI_MRN"].isin(prostate_mrns_pd)
].reset_index(drop=True)


labs_df_pd = pd.read_csv(
    DATA_PATH / "OUTPT_LAB_RESULTS_LABS.csv",
    usecols=labs_cols,
    engine="pyarrow",
    dtype_backend="pyarrow",
    dtype={
        "DFCI_MRN": "string[pyarrow]",
    },
)

labs_df_pd = labs_df_pd.loc[
    labs_df_pd["DFCI_MRN"].isin(prostate_mrns_pd)
].reset_index(drop=True)

pandas_seconds = perf_counter() - pandas_start

print(f"pandas total time: {pandas_seconds:,.2f} seconds")
print(f"Unique prostate MRNs: {len(prostate_mrns_pd):,}")
print(f"Medication rows:       {len(meds_df_pd):,}")
print(f"Status rows:           {len(status_df_pd):,}")
print(f"Lab rows:              {len(labs_df_pd):,}")

In [ ]:
benchmark_df = pd.DataFrame({
    "method": ["Polars", "pandas/PyArrow"],
    "seconds": [polars_seconds, pandas_seconds],
})

benchmark_df["minutes"] = benchmark_df["seconds"] / 60

display(benchmark_df)

print(
    f"Polars speed relative to pandas: "
    f"{pandas_seconds / polars_seconds:.2f}x"
)

In [ ]:
validation_df = pd.DataFrame({
    "dataset": [
        "Prostate MRNs",
        "Medications",
        "Status",
        "Labs",
    ],
    "polars_count": [
        prostate_mrns_pl.height,
        meds_df_pl.height,
        status_df_pl.height,
        labs_df_pl.height,
    ],
    "pandas_count": [
        len(prostate_mrns_pd),
        len(meds_df_pd),
        len(status_df_pd),
        len(labs_df_pd),
    ],
})

validation_df["matches"] = (
    validation_df["polars_count"]
    == validation_df["pandas_count"]
)

display(validation_df)